# Dasymetric Mapping

Dasymetric mapping redistributes aggregate zone-level data (like population per census tract) onto a finer raster grid using ancillary weight information (land cover, nighttime lights, etc.).

This is the spatial inverse of `zonal.stats`: instead of aggregating pixel values into zone summaries, we spread zone-level totals back across pixels, weighted by auxiliary data.

`xrspatial.disaggregate` supports three methods:
- **`'binary'`** -- split value equally among nonzero-weight pixels
- **`'weighted'`** (default) -- distribute proportionally to weight values
- **`'limiting_variable'`** -- three-class dasymetric with density caps (numpy-only)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial import disaggregate

## Generate Synthetic Data

We create a small grid of census-tract-like zones, assign population values per zone, and build a land-cover-based weight raster.

In [ ]:
# 10x10 zone raster with 4 "census tracts"
zones_data = np.ones((10, 10), dtype=np.float64)
zones_data[:5, :5] = 1
zones_data[:5, 5:] = 2
zones_data[5:, :5] = 3
zones_data[5:, 5:] = 4

zones = xr.DataArray(zones_data, dims=['y', 'x'])

# Population per tract
population = {1: 1000.0, 2: 500.0, 3: 2000.0, 4: 300.0}

# Ancillary weight raster -- simulates land cover suitability
np.random.seed(42)
weight_data = np.random.rand(10, 10).astype(np.float64)
# Make some areas uninhabitable (zero weight)
weight_data[0:2, 0:2] = 0.0  # water body in tract 1
weight_data[7:9, 7:9] = 0.0  # park in tract 4

weight = xr.DataArray(weight_data, dims=['y', 'x'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
zones.plot(ax=axes[0], cmap='Set2')
axes[0].set_title('Zones (Census Tracts)')
weight.plot(ax=axes[1], cmap='YlGn')
axes[1].set_title('Weight (Land Cover)')
plt.tight_layout()
plt.show()

## Binary Method

The `'binary'` method binarizes the weight raster (nonzero becomes 1, zero stays 0) and splits each zone's value equally among its nonzero pixels.

In [ ]:
result_binary = disaggregate(zones, population, weight, method='binary')

fig, ax = plt.subplots(figsize=(6, 5))
result_binary.plot(ax=ax, cmap='YlOrRd')
ax.set_title('Binary Dasymetric Result')
plt.tight_layout()
plt.show()

## Weighted Method

The `'weighted'` method (default) distributes each zone's value proportionally to the weight values:

$$\text{pixel} = \text{zone\_value} \times \frac{\text{pixel\_weight}}{\sum_{\text{zone}} \text{weights}}$$

In [ ]:
result_weighted = disaggregate(zones, population, weight, method='weighted')

fig, ax = plt.subplots(figsize=(6, 5))
result_weighted.plot(ax=ax, cmap='YlOrRd')
ax.set_title('Weighted Dasymetric Result')
plt.tight_layout()
plt.show()

## Conservation Property

A key property of dasymetric mapping: the sum of output pixel values within each zone should equal the original zone total.

In [ ]:
for zid, expected in population.items():
    actual = float(np.nansum(result_weighted.values[zones_data == zid]))
    print(f'Zone {zid}: expected={expected:.1f}, actual={actual:.1f}, '
          f'diff={abs(expected - actual):.2e}')

## Comparing Methods

The binary method produces uniform density within each zone (among habitable pixels), while the weighted method produces spatially varying density.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

result_binary.plot(ax=axes[0], cmap='YlOrRd', vmin=0,
                   vmax=float(np.nanmax(result_weighted.values)))
axes[0].set_title('Binary')

result_weighted.plot(ax=axes[1], cmap='YlOrRd', vmin=0,
                     vmax=float(np.nanmax(result_weighted.values)))
axes[1].set_title('Weighted')

plt.tight_layout()
plt.show()

## Limiting Variable Method

The `'limiting_variable'` method applies per-class density caps and redistributes overflow iteratively. This is useful when you know certain land cover types have maximum population densities. Currently only available for numpy arrays.

In [ ]:
result_lv = disaggregate(zones, population, weight,
                         method='limiting_variable')

fig, ax = plt.subplots(figsize=(6, 5))
result_lv.plot(ax=ax, cmap='YlOrRd')
ax.set_title('Limiting Variable Result')
plt.tight_layout()
plt.show()